<a href="https://colab.research.google.com/github/Erick-dev-dot/SPI-Analysis-2015-2025-Colab-Notebook/blob/main/SPIs_2015_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee

In [ ]:
ee.Authenticate()

In [ ]:
# Initializing Earth Engine by specifying Google Cloud Project ID

ee.Initialize(project='gee-python-api-498913')

1. **Importing necessary libraries.**

In [ ]:
# Importing necessary libraries

import ee
import geemap
import pandas as pd
import numpy as np
import scipy.stats as stats

2. **Defining necessary parameters.**

In [ ]:
# DEFINE SPATIAL & TEMPORAL PARAMETERS

# 1. Kenya Boundary
kenya = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017') \
    .filter(ee.Filter.eq('country_na', 'Kenya'))

# 2. Temporal bounds
start_history = '1981-01-01'  # Must start here to calculate accumulated periods
end_target = '2025-12-31'

baseline_start_year = 1981
baseline_end_year = 2010

target_start_year = 2015
target_end_year = 2025


### **3. Visualizing the Kenya boundary on an interactive map**

In [ ]:
# 1. Load the Large Scale International Boundary (LSIB) dataset
countries = ee.FeatureCollection('USDOS/LSIB_SIMPLE/2017')

# 2. Filter for Kenya using the country name
kenya_boundary = countries.filter(ee.Filter.eq('country_na', 'Kenya'))

# Create an interactive map centered over Kenya using an alternative basemap
# Options include: 'ROADMAP', 'SATELLITE', 'TERRAIN', or 'HYBRID'
Map = geemap.Map(center=[0.0236, 37.9062], zoom=6, basemap='ROADMAP')

# Add the Kenya boundary layer to the map
Map.addLayer(kenya_boundary, {'color': 'red', 'fillColor': '00000000'}, 'Kenya Boundary')

# Display the map
Map


### **4. Visualizing the time series of monthly precipitation for Kenya**

In [ ]:
# Function to extract monthly precipitation for Kenya
def extract_monthly_precipitation(image):
    date = ee.Date(image.get('system:time_start'))
    year = date.get('year')
    month = date.get('month')

    # Reducing the precipitation band over the Kenya boundary
    # Using ee.Reducer.mean() to get the average precipitation across Kenya for the month

    mean_precipitation = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=kenya.geometry(),
        scale=55660, # CHIRPS pentad resolution is approximately 5.5km
        maxPixels=1e10
    ).get('precipitation')

    return ee.Feature(None, {
        'date': date.format('YYYY-MM-dd'),
        'year': year,
        'month': month,
        'precipitation': mean_precipitation
    })

# Mapping the function over the monthly images to get a feature collection of results
monthly_precipitation_fc = monthly_images.map(extract_monthly_precipitation)

# Converting the FeatureCollection to a list of dictionaries for client-side processing
precipitation_list = monthly_precipitation_fc.getInfo()['features']

# Converting to Pandas DataFrame
data = []
for feature in precipitation_list:
    properties = feature['properties']
    if properties['precipitation'] is not None: # Filter out null values
        data.append({
            'date': properties['date'],
            'precipitation': properties['precipitation']
        })

df_precipitation = pd.DataFrame(data)
df_precipitation['date'] = pd.to_datetime(df_precipitation['date'])

display(df_precipitation.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting the time series
plt.figure(figsize=(15, 7))
sns.lineplot(x='date', y='precipitation', data=df_precipitation)
plt.title('Monthly Precipitation Time Series for Kenya (1981-2025)')
plt.xlabel('Date')
plt.ylabel('Average Monthly Precipitation (mm)')
plt.grid(True)
plt.tight_layout()
plt.show()

The plot shows the average monthly precipitation in millimeters from 1981 to 2025. The seasonal patterns and inter-annual variability in precipitation over this period can be observed. The table above the plot displays the first five rows of the dataframe containing the date and precipitation values.



### **5. Identifying months with highest and lowest precipitation**

In [ ]:
# Extracting month from the date
df_precipitation['month'] = df_precipitation['date'].dt.month

# Calculating average precipitation per month
monthly_avg_precipitation = df_precipitation.groupby('month')['precipitation'].mean().reset_index()

# Mapping month numbers to names for better readability
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_avg_precipitation['month_name'] = monthly_avg_precipitation['month'].map(lambda x: month_names[x-1])

# Identifying the month with the highest average precipitation
max_precipitation_month = monthly_avg_precipitation.loc[monthly_avg_precipitation['precipitation'].idxmax()]

# Identifying the month with the lowest average precipitation
min_precipitation_month = monthly_avg_precipitation.loc[monthly_avg_precipitation['precipitation'].idxmin()]

print(f"Month with highest average precipitation: {max_precipitation_month['month_name']} ({max_precipitation_month['precipitation']:.2f} mm)")
print(f"Month with lowest average precipitation: {min_precipitation_month['month_name']} ({min_precipitation_month['precipitation']:.2f} mm)")

display(monthly_avg_precipitation)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting the average monthly precipitation
plt.figure(figsize=(12, 6))
sns.barplot(x='month_name', y='precipitation', data=monthly_avg_precipitation, palette='viridis')
plt.title('Average Monthly Precipitation in Kenya (1981-2025)')
plt.xlabel('Month')
plt.ylabel('Average Precipitation (mm)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Based on the analysis, April is the month with the highest average precipitation, at approximately 129.69 mm. Conversely, February experiences the lowest average precipitation, with about 18.90 mm. The bar plot visually confirms these trends, showing a clear peak in April and a trough in February, with a secondary peak in November. The table above displays the average precipitation for each month.

### **6. Calculating the standard deviation of monthly precipitation**

In [ ]:
# Calculating the std of precipitation for each month
monthly_std_precipitation = df_precipitation.groupby('month')['precipitation'].std().reset_index()

# Mapping month numbers to names for better readability
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_std_precipitation['month_name'] = monthly_std_precipitation['month'].map(lambda x: month_names[x-1])

print("Standard Deviation of Monthly Precipitation:")
display(monthly_std_precipitation)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting the std of monthly precipitation
plt.figure(figsize=(12, 6))
sns.barplot(x='month_name', y='precipitation', data=monthly_std_precipitation, palette='coolwarm')
plt.title('Standard Deviation of Monthly Precipitation in Kenya (1981-2025)')
plt.xlabel('Month')
plt.ylabel('Standard Deviation of Precipitation (mm)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

The bar chart shows that November has the highest variability in precipitation, with a standard deviation of approximately 50.21 mm, followed closely by April and March. Conversely, September, July, August, and June exhibit the lowest variability, indicating more consistent precipitation levels during these months. The table above displays the standard deviation for each month. This information is crucial for understanding the predictability and consistency of rainfall throughout the year in Kenya.

Consistency: A low standard deviation shows that data points are clustered very close to the mean, indicating high consistency.

Variability: A high standard deviation shows that data points are spread out widely, indicating high volatility or diversity.

**7. Extracting Monthly CHIRPS Data**

In [ ]:
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD') \
    .filterDate(start_history, end_target) \
    .filterBounds(kenya) \
    .select('precipitation')

years = ee.List.sequence(1981, 2025)
months = ee.List.sequence(1, 12)

def make_monthly(y):
    return months.map(lambda m: chirps.filter(ee.Filter.calendarRange(y, y, 'year'))
                      .filter(ee.Filter.calendarRange(m, m, 'month'))
                      .sum()
                      .set('year', y)
                      .set('month', m)
                      .set('system:time_start', ee.Date.fromYMD(y, m, 1).millis()))

monthly_images = ee.ImageCollection.fromImages(years.map(make_monthly).flatten())


**8. Computing Rolling Accumulations (SPI TIMESCALES)**

In [ ]:
# Converting collection to a list for sequential rolling window analysis
monthly_list = monthly_images.sort('system:time_start').toList(monthly_images.size())

def create_accumulated_collection(timescale):
    accumulated_list = ee.List.sequence(timescale - 1, monthly_images.size().subtract(1)).map(
        lambda index: ee.ImageCollection(monthly_list.slice(ee.Number(index).subtract(timescale - 1), ee.Number(index).add(1)))
                        .sum()
                        .copyProperties(monthly_list.get(index), ['year', 'month', 'system:time_start'])
    )
    return ee.ImageCollection.fromImages(accumulated_list)

# Generating accumulated rainfall layers
acc_3 = create_accumulated_collection(3)
acc_6 = create_accumulated_collection(6)
acc_12 = create_accumulated_collection(12)


**9. DATA EXTRACTION & CLIENT-SIDE SPI GAMMA PROCESSING**

In [ ]:
def calculate_gamma_spi(df, baseline_start, baseline_end, target_start, target_end):
    """
    Fits a Gamma distribution to the baseline years (1981-2010) per calendar month,
    and transforms the target years (2015-2025) rainfall into SPI values.
    """
    spi_results = []

    # Split baseline and target dataframes
    baseline_df = df[(df['year'] >= baseline_start) & (df['year'] <= baseline_end)]
    target_df = df[(df['year'] >= target_start) & (df['year'] <= target_end)].copy()

    target_df['SPI'] = np.nan

    # Processing month-by-month to capture seasonality
    for m in range(1, 13):
        b_month = baseline_df[baseline_df['month'] == m]['precipitation'].values
        t_month_mask = target_df['month'] == m
        t_month_vals = target_df[t_month_mask]['precipitation'].values

        if len(b_month) < 10:
            continue # Ensure enough data exists

        # Fitting Gamma distribution parameters (shape, location, scale) to baseline data
        # Fixing location to 0 is a standard meteorological best practice for precipitation
        shape, loc, scale = stats.gamma.fit(b_month, floc=0)

        # Calculating Cumulative Probability (CDF) for target years
        cdf = stats.gamma.cdf(t_month_vals, shape, loc, scale)

        # Avoid mathematical infinity errors at extreme bounds [0, 1]
        cdf = np.clip(cdf, 0.0001, 0.9999)

        # Transforming probabilities into Standard Normal Distribution values (SPI)
        spi_vals = stats.norm.ppf(cdf)
        target_df.loc[t_month_mask, 'SPI'] = spi_vals

    return target_df

# Example Data Check: Summarize Kenya's National Average to a Dataframe
def extract_regional_average(collection):
    def get_mean(img):
        mean_dict = img.reduceRegion(reducer=ee.Reducer.mean(), geometry=kenya, scale=5000)
        return ee.Feature(None, {
            'year': img.get('year'),
            'month': img.get('month'),
            'precipitation': mean_dict.get('precipitation')
        })
    features = collection.map(get_mean).getInfo()['features']
    return pd.DataFrame([f['properties'] for f in features])

# Run the pipeline for SPI-3, SPI-6, and SPI-12
print("Extracting regional timeseries from Google Earth Engine...")
df_3 = extract_regional_average(acc_3)
df_6 = extract_regional_average(acc_6)
df_12 = extract_regional_average(acc_12)

print("Computing Gamma Distributed SPI values...")
spi_3_final = calculate_gamma_spi(df_3, baseline_start_year, baseline_end_year, target_start_year, target_end_year)
spi_6_final = calculate_gamma_spi(df_6, baseline_start_year, baseline_end_year, target_start_year, target_end_year)
spi_12_final = calculate_gamma_spi(df_12, baseline_start_year, baseline_end_year, target_start_year, target_end_year)

# View sample of target analysis output
print("\n--- Example Output for SPI-3 (2015-2025 Target Window) ---")
print(spi_3_final[['year', 'month', 'precipitation', 'SPI']].head(12))


**10. SPI-3, SPI-6, and SPI-12 Individual Charts**

In [ ]:
import matplotlib.pyplot as plt

def plot_individual_spi(spi_df, timescale_title):

    # Creating a proper datetime index for plotting chronological timelines

    spi_df = spi_df.copy()
    spi_df['date'] = pd.to_datetime(spi_df['year'].astype(str) + '-' + spi_df['month'].astype(str) + '-01')
    spi_df = spi_df.sort_values('date')

    plt.figure(figsize=(12, 4))

    # Plot the base SPI line
    plt.plot(spi_df['date'], spi_df['SPI'], color='black', linewidth=1, label=f'SPI-{timescale_title}')

    # Fill Wet anomalies (SPI > 0) in Blue
    plt.fill_between(spi_df['date'], spi_df['SPI'], 0, where=(spi_df['SPI'] >= 0),
                     color='royalblue', alpha=0.7)

    # Fill Dry anomalies (SPI < 0) in Red
    plt.fill_between(spi_df['date'], spi_df['SPI'], 0, where=(spi_df['SPI'] < 0),
                     color='crimson', alpha=0.7)

    # Formatting thresholds for extreme conditions
    plt.axhline(0, color='black', linestyle='-', linewidth=0.8)
    plt.axhline(1.5, color='blue', linestyle='--', linewidth=0.8, alpha=0.5, label='Severely Wet (1.5)')
    plt.axhline(-1.5, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='Severely Dry (-1.5)')

    plt.title(f'Kenya National Standardized Precipitation Index (SPI-{timescale_title}): 2015 - 2025')
    plt.ylabel('SPI Value')
    plt.xlabel('Timeline')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.show()

# Execute separate plots for each index scale
plot_individual_spi(spi_3_final, '3')
plot_individual_spi(spi_6_final, '6')
plot_individual_spi(spi_12_final, '12')


The baseline period for this SPI analysis is from 1981 to 2010.

**11. Spatial Precipitation Maps (2015, 2020, 2022, 2025)**

In [ ]:
import geemap

# Initialize a clear interactive map instance centered on Kenya
# use_google_maps=False prevents geemap from attempting to load Google basemaps implicitly.
Map = geemap.Map(center=[0.0236, 37.9062], zoom=6, use_google_maps=False, basemap='ROADMAP')

# Note: With 'ROADMAP' as basemap, 'Map.clear_layers()' and 'Map.add_basemap()' are not needed after initialization
# as the basemap is set directly in the constructor.

# Define visual style for Rainfall (Brown/Yellow = Dry, Blue/Purple = Very Wet)
precip_vis = {
    'min': 200,      # Low annual rain in millimeters (Arid regions like Turkana)
    'max': 2000,     # High annual rain in millimeters (Highlands/Lake Basin)
    'palette': ['#f5eccb', '#e1ca96', '#9bc694', '#63a375', '#337a7b', '#1e4860', '#0b1d3a']
}

# Target Years requested
target_years = [2015, 2020, 2022, 2025]

for year in target_years:
    # Set temporal bounds for the given calendar year
    year_start = f'{year}-01-01'
    year_end = f'{year}-12-31'

    # Filter collection and calculate the sum of total rainfall over those 12 months
    annual_total = ee.ImageCollection('UCSB-CHG/CHIRPS/PENTAD') \
        .filterDate(year_start, year_end) \
        .filterBounds(kenya) \
        .select('precipitation') \
        .sum() \
        .clip(kenya)

    # Inject layer into the map panel
    Map.addLayer(annual_total, precip_vis, f'Kenya Total Rainfall ({year})')

# Render Map layer selectors
Map.add_layer_control()
Map

From the rainfall visual map above, 2022 is depicted to have received the lowest rainfall among the the selected epoch years.